In [ ]:
# Imports
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import random, os, json
from itertools import product
from collections import defaultdict

from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data

import numpy as np
import umap
import matplotlib.pyplot as plt

MODE = "full" # "full" for full implementation of antropic strategy, "standard" for standard baseline
MODEL_NAME = "./codebert-base-mlm"

In [ ]:
# Load dataset

f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
ai_dataset = [json.loads(line) for line in f.readlines()]
f.close()

In [ ]:
# Getting Data activations


model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

inputs = [s["text"] for s in ai_dataset]
tokens_id = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
activations =  model(**tokens_id).hidden_states

num_samples, num_tokens, _ = activations[layer].shape
tokens = [{"tokens_str": ["CLS"] + tokenizer.tokenize(s["text"], truncation=True,padding="max_length", max_length=num_tokens) + ["</s>"], "class": s["class"]} for s in ai_dataset]

print("Done gathering inputs")

In [ ]:
class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
# Utility functions for SAE analysis
def calculate_sae_loss(x, W_enc, b_enc, W_dec, b_dec, lambd):
    """
    Calcola la loss function per uno Sparse AutoEncoder come specificato nell'immagine.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).
        b_dec (torch.Tensor): Bias del decoder, di forma (D,).
        lambd (float): Parametro di regolarizzazione.

    Returns:
        torch.Tensor: Il valore scalare della loss.
    """
    
    # Formula: f(x) = ReLU(W_enc * x + b_enc)
    f_x = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    
    # Formula: x_hat = W_dec * f(x) + b_dec
    x_hat = torch.matmul(f_x, W_dec.T) + b_dec
    
    # --- 1. Reconstruction Loss ---
    reconstruction_loss = torch.mean(torch.sum((x - x_hat)**2, dim=1))

    # --- 2. Termine di Sparsità/Regolarizzazione ---
    # Formula: lambda * E_x [ sum_i f_i(x) * ||W_dec[:,i]||_2 ]

    # a. Calcola la norma L2 (Euclidea) di ciascuna colonna di W_dec (corrispondente a W_dec[:,i])
    # torch.linalg.norm(W_dec, dim=0, ord=2) calcola ||W_dec[:,i]||_2 per tutte le i.
    # Risultato: un tensore di dimensione [F]
    decoder_l2_norms = torch.linalg.norm(W_dec, dim=0, ord=2)

    # b. Calcola f_i(x) * ||W_dec[:,i]||_2
    # encoded (Batch_size x F) viene moltiplicato per decoder_l2_norms (F).
    # Il broadcasting assicura che ogni f_i(x) sia moltiplicato per il suo rispettivo ||W_dec[:,i]||_2.
    weighted_activations = f_x * decoder_l2_norms

    # c. Calcola la somma su tutte le feature (sum_i) e il Valore Atteso (E_x tramite mean sul batch)
    # Somma su tutte le feature (dim=1)
    sum_of_weighted_activations = torch.sum(weighted_activations, dim=1)
    
    # Calcola il valore atteso sul batch (media)
    mean_regularization_term = torch.mean(sum_of_weighted_activations)

    # Calcola il termine di regolarizzazione
    regularization_loss = lambd * mean_regularization_term

    # --- 3. Perdita Totale ---
    total_loss = reconstruction_loss + regularization_loss
    
    return total_loss


def get_feature_directions(W_dec):
    """
    Calcola i vettori di direzione delle feature a partire dalla matrice dei pesi del decoder.

    Args:
        W_dec (torch.Tensor): Pesi del decoder di forma (D, F), dove
                              D è la dimensione residua e F la dimensione delle feature.

    Returns:
        torch.Tensor: I vettori delle feature (direzioni normalizzate), di forma (D, F).
    """
    # Calcola la norma L2 di ogni colonna (dim=0) della matrice W_dec.
    # Aggiunge 1e-8 per evitare divisioni per zero.
    norms = torch.linalg.norm(W_dec, dim=0)
    
    # Normalizza ogni colonna (vettore di feature) dividendo per la sua norma.
    feature_directions = W_dec / (norms + 1e-8)
    
    return feature_directions

def get_feature_activations(x, W_enc, b_enc, W_dec):
    """
    Calcola le attivazioni delle feature per un dato input x.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).

    Returns:
        torch.Tensor: Le attivazioni delle feature, di forma (batch_size, F).
    """
    # Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    f_x = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    
    norm_W_dec = torch.linalg.norm(W_dec, dim=0)
    
    feature_activations = f_x * norm_W_dec
    
    return feature_activations

def init_decoder_weights(input_dim, hidden_dim, l2_norm=0.1):
    # Crea una matrice random di shape (input_dim, hidden_dim)
    W_d = torch.randn(input_dim, hidden_dim)
    # Normalizza ogni colonna a norma L2 = l2_norm
    W_d = W_d / W_d.norm(dim=0, keepdim=True) * l2_norm
    return W_d

In [ ]:
# Analyzing feature statistics for all saes

hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32] 
layer = 10
beta = 5.0
lr = 5e-5
input_dim = 768
feature_directions_dict = {}
feature_activations_dict = {}

for hidden_dim in hidden_dims:
    sae = SAE(input_dim, hidden_dim)
    sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}_{MODE}.pt"))
    encoder_weights = sae.encoder[0].weight
    decoder_weights = sae.decoder[0].weight
    feature_directions = get_feature_directions(decoder_weights)
    feature_activations = get_feature_activations(activations[layer], encoder_weights, sae.encoder[0].bias, decoder_weights)
    feature_directions_dict[hidden_dim] = feature_directions
    feature_activations_dict[hidden_dim] = feature_activations

        
    nonzero_counts = (feature_activations != 0).sum(dim=2)  # shape: (100, 512)
    flattened_nonzero_counts = nonzero_counts.flatten()
    sparsity = flattened_nonzero_counts.float().mean().item()
    print(f"[Dim {hidden_dim}] Sparsità media (numero di feature attive per singolo token): {sparsity:.4f}")

    mean_activations = feature_activations.mean(dim=(0,1)) # shape: (hidden_dim)
    live_features = (mean_activations!=0).float().mean().item()
    print(f"[Dim {hidden_dim}] Valore medio feature attive: {live_features:.4f}")

    # Calcola per ogni feature (sull'ultima dimensione) se è sempre zero su tutti i token di tutti i sample

    dead_features = (feature_activations == 0).all(dim=(0, 1)).sum().item()
    print(f"[Dim {hidden_dim}] Numero di dead features (mai attivate): {dead_features} su {len(mean_activations)}")

    top_features = torch.topk(mean_activations, k=10)
    print(f"[Dim {hidden_dim}] Top 10 feature (concetti) più attive:")
    for idx, value in zip(top_features.indices.tolist(), top_features.values.tolist()):
        print(idx, value)
        print(f"Feature {idx}: attivazione media = {value:.4f}")
        

import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, n_neighbors=15, metric="cosine", min_dist=0.01)
colors = ['red', 'orange', 'green', 'blue', 'purple', 'black']
plt.figure(figsize=(10, 8))

min_hd = min(hidden_dims)
base_size = 20
# scala la dimensione dei punti in modo decrescente al crescere di hidden_dim
sizes = [max(5, base_size * (min_hd / hd) ** 0.5) for hd in hidden_dims]

for i, hidden_dim in enumerate(hidden_dims):
    fd = feature_directions_dict[hidden_dim].detach().cpu().numpy().T  # shape: (hidden_dim, input_dim)
    embedding = reducer.fit_transform(fd)
    plt.scatter(
        embedding[:, 0], embedding[:, 1],
        label=f"hidden_dim={hidden_dim}",
        alpha=0.6, s=sizes[i], color=colors[i % len(colors)]
    )

plt.title("UMAP 2D delle feature_directions per diversi hidden_dim")
plt.legend()
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

In [ ]:
class_common = {}   # intersezione (feature attive in TUTTI i sample della classe)
class_any = {}      # unione (feature attive in ALMENO un sample della classe)

# Analysis of each sample in each sae

for hidden_dim in hidden_dims:
    sae = SAE(input_dim, hidden_dim)
    sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt"))
    encoder_weights = sae.encoder[0].weight
    decoder_weights = sae.decoder[0].weight
    feature_directions = feature_directions_dict[hidden_dim]
    feature_activations = feature_activations_dict[hidden_dim]
    
    class_common[hidden_dim] = {}
    class_any[hidden_dim] = {}

    
    output = []
    num_samples, num_tokens, hidden_dim = feature_activations.shape
            
    for sample_idx in tqdm(range(num_samples)):

        cls = tokens[sample_idx]["class"]

        # attivazioni del sample: shape (num_tokens, hidden_dim)
        sample_acts = feature_activations[sample_idx]

        # feature attive su tutti i token del sample / su almeno un token del sample
        active_features = set()
        # più veloce: trova gli indici delle feature non-zero su almeno un token
        # sample_acts: shape (num_tokens, hidden_dim)
        mask = (sample_acts != 0).any(dim=0)  # (hidden_dim,) boolean tensor
        active_idx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
        active_features = set(active_idx.tolist()) if active_idx.numel() > 0 else set()
        
        if class_common[hidden_dim].get(cls) is None:
            class_common[hidden_dim][cls] = set(active_features)
        else:
            class_common[hidden_dim][cls] = class_common[hidden_dim][cls].intersection(active_features)
            
        if class_any[hidden_dim].get(cls) is None:
            class_any[hidden_dim][cls] = set(active_features)
        else:
            class_any[hidden_dim][cls] = class_any[hidden_dim][cls].union(active_features)


In [ ]:
# Print analysis results
for hidden_dim in hidden_dims:
        print(f"\n=== hidden_dim = {hidden_dim} ===")
        any_dict = class_any[hidden_dim]
        common_dict = class_common[hidden_dim]
        # precompute union per hidden_dim for unique calculation
        all_classes = list(any_dict.keys())
        for cls in sorted(all_classes):
                any_set = any_dict.get(cls, set())
                common_set = common_dict.get(cls, set())

                # unique to this class (in its union but in no other class union)
                other_union = set().union(*(any_dict[c] for c in all_classes if c != cls))
                unique_set = any_set - other_union

                # compute how many features are active in exactly one sample of this class
                # uses feature_activations_dict[hidden_dim] (shape: num_samples x num_tokens x hidden_dim)
                fa = feature_activations_dict[hidden_dim]  # tensor
                # find sample indices belonging to this class
                sample_idxs = [i for i, t in enumerate(tokens) if t["class"] == cls]
                if len(sample_idxs) == 0:
                        activated_once_count = 0
                else:
                        # slice and compute per-sample whether each feature is active (any token)
                        class_acts = fa[sample_idxs]  # (n_samples_cls, n_tokens, hidden_dim)
                        per_sample_active = (class_acts != 0).any(dim=1).float()  # (n_samples_cls, hidden_dim)
                        active_counts_per_feature = per_sample_active.sum(dim=0)  # (hidden_dim,)
                        activated_once_count = int((active_counts_per_feature == 1).sum().item())

                print(f"\nClasse = {cls}")
                print(f"  - Feature sempre attive (intersection across samples): {len(common_set)}")
                print(f"    Example indices (up to 20): {sorted(list(common_set))[:20]}")
                print(f"  - Feature attivate almeno una volta (union across samples): {len(any_set)}")
                print(f"    Example indices (up to 20): {sorted(list(any_set))[:20]}")
                print(f"  - Feature uniche alla classe (presenti solo in questa classe): {len(unique_set)}")
                print(f"    Example indices (up to 20): {sorted(list(unique_set))[:20]}")
                print(f"  - Feature attivate esattamente in una singola sample della classe: {activated_once_count}")


In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

for hidden_dim in hidden_dims:
    fa = feature_activations_dict[hidden_dim]  # tensor shape: (num_samples, num_tokens, hidden_dim)
    num_samples, num_tokens, _ = fa.shape
    out_path = f"outputs/sae_layer_{layer}_hiddim_{hidden_dim}_{MODE}_tokens.jsonl"
    with open(out_path, "w", encoding="utf-8") as fout:
        for sample_idx in tqdm(range(num_samples), desc=f"export hiddim={hidden_dim}"):
            sample_tokens_str = tokens[sample_idx]["tokens_str"]
            cls = tokens[sample_idx]["class"]

            tokens_json = []
            for tok_idx in range(num_tokens):
                # safe token string access (fall back to empty string if index out of range)
                token_str = sample_tokens_str[tok_idx] if tok_idx < len(sample_tokens_str) else ""

                vec = fa[sample_idx, tok_idx].cpu()
                nonzero_idx = torch.nonzero(vec != 0, as_tuple=False).squeeze(-1)
                activations = []
                if nonzero_idx.numel() > 0:
                    for fid in nonzero_idx.tolist():
                        activations.append((int(fid), float(vec[fid].item())))  # tuple -> will be serialized as array

                tokens_json.append({
                    "token_idx": tok_idx,
                    "token_str": token_str,
                    "activations": activations
                })

            sample_json = {
                "sample_index": sample_idx,
                "tokens": tokens_json,
                "class": cls
            }
            fout.write(json.dumps(sample_json) + "\n")
    print(f"Wrote {out_path}")